# **03. Data Splitting and Baseline Models**

This notebook establishes baseline performance for category and priority prediction. The goal is to compare simple, explainable models before moving to the final training pipeline.

We use validation results for model selection. The test split is held back for final evaluation.

# **1. Environment Setup**

The notebook imports the same preprocessing and evaluation functions used by the project implementation.

In [1]:
# Import libraries and reusable project functions.
from pathlib import Path
import sys

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'customer_support_ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from customer_support_ai.config import COMPATIBLE_DATASET_PATHS
from customer_support_ai.data_preprocessing import (
    build_modelling_frame,
    load_project_dataset,
    make_train_validation_test_split,
)
from customer_support_ai.evaluate import evaluate_predictions
from customer_support_ai.train import build_models

# **2. Load Model-Ready Data**

We rebuild the modelling frame so this notebook always reflects the current preprocessing code.

In [2]:
# Load the compatible merged dataset and create model-ready features.
raw = load_project_dataset(COMPATIBLE_DATASET_PATHS)
frame, category_col, priority_col = build_modelling_frame(raw)

print(f'Model rows: {len(frame):,}')
print(f'Metadata columns used: {frame.attrs.get("metadata_columns")}')
frame[['model_text', category_col, priority_col]].head()

Model rows: 44,275
Metadata columns used: ['type', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']


,model_text,queue,priority
0,wesentlicher sicherheitsvorfall sehr geehrtes ...,Technical Support,high
1,account disruption dear customer support team ...,Technical Support,high
2,query about smart home system integration feat...,Returns and Exchanges,medium
3,inquiry regarding invoice details dear custome...,Billing and Payments,low
4,question about marketing agency software compa...,Sales and Pre-Sales,medium


# **3. Train, Validation, and Test Splits**

The split design is 70% training, 15% validation, and 15% testing. Validation is used to compare models; testing is saved for the final selected model.

In [3]:
# Create splits for the category task and show split sizes.
category_train, category_validation, category_test = make_train_validation_test_split(frame, category_col)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(category_train), len(category_validation), len(category_test)],
})
split_summary['share'] = split_summary['rows'] / len(frame)
split_summary

,split,rows,share
0,train,30991,0.699966
1,validation,6642,0.150017
2,test,6642,0.150017


In [4]:
# Confirm that category label proportions are similar across splits.
category_split_balance = pd.concat({
    'train': category_train[category_col].value_counts(normalize=True),
    'validation': category_validation[category_col].value_counts(normalize=True),
    'test': category_test[category_col].value_counts(normalize=True),
}, axis=1).round(4)

category_split_balance

,train,validation,test
queue,,,
Technical Support,0.2958,0.2958,0.2957
Product Support,0.1831,0.1832,0.1832
Customer Service,0.1523,0.1522,0.1522
IT Support,0.1183,0.1183,0.1183
Billing and Payments,0.0987,0.0986,0.0986
Returns and Exchanges,0.0498,0.0498,0.0498
Service Outages and Maintenance,0.0386,0.0385,0.0387
Sales and Pre-Sales,0.0315,0.0315,0.0315
Human Resources,0.0181,0.0181,0.0181


# **4. Baseline Model Set**

We include a majority-class dummy baseline so the real models can be judged against a simple reference point. The main AI baselines are TF-IDF with Naive Bayes, Logistic Regression, and Linear SVM.

In [5]:
# Define baseline models for comparison.
def build_notebook_models():
    models = {
        'dummy_most_frequent': Pipeline([
            ('tfidf', TfidfVectorizer(max_features=5000)),
            ('model', DummyClassifier(strategy='most_frequent')),
        ])
    }
    models.update(build_models())
    return models

list(build_notebook_models().keys())

['dummy_most_frequent', 'logistic_regression', 'linear_svm', 'naive_bayes']

# **5. Category Prediction Baselines**

Category prediction uses the support queue as the routing label. Notebook 02 showed useful tag and text variation, so the model should perform better than the previous dataset.

In [6]:
# Train category models and evaluate on validation data.
category_rows = []
for name, model in build_notebook_models().items():
    model.fit(category_train['model_text'], category_train[category_col])
    predictions = model.predict(category_validation['model_text'])
    result = evaluate_predictions(category_validation[category_col], predictions, 'category', name, 'validation')
    category_rows.append({key: value for key, value in result.items() if key != 'classification_report'})

category_results = pd.DataFrame(category_rows).sort_values('macro_f1', ascending=False)
category_results

,task,model,split,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
2,category,linear_svm,validation,0.515507,0.477816,0.513708,0.493164,0.516123
1,category,logistic_regression,validation,0.429088,0.381896,0.475032,0.408271,0.434263
3,category,naive_bayes,validation,0.391749,0.570440,0.209067,0.201916,0.331944
0,category,dummy_most_frequent,validation,0.295845,0.029584,0.100000,0.045661,0.135084


# **6. Priority Prediction Baselines**

Priority has four balanced classes. A useful model should beat the dummy baseline and show stronger macro F1 than random guessing.

In [7]:
# Create splits for the priority task.
priority_train, priority_validation, priority_test = make_train_validation_test_split(frame, priority_col)

priority_rows = []
for name, model in build_notebook_models().items():
    model.fit(priority_train['model_text'], priority_train[priority_col])
    predictions = model.predict(priority_validation['model_text'])
    result = evaluate_predictions(priority_validation[priority_col], predictions, 'priority', name, 'validation')
    priority_rows.append({key: value for key, value in result.items() if key != 'classification_report'})

priority_results = pd.DataFrame(priority_rows).sort_values('macro_f1', ascending=False)
priority_results

,task,model,split,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
2,priority,linear_svm,validation,0.567600,0.547937,0.549990,0.548877,0.568009
1,priority,logistic_regression,validation,0.540349,0.526236,0.536869,0.527821,0.543624
3,priority,naive_bayes,validation,0.494279,0.492498,0.419667,0.389717,0.451140
0,priority,dummy_most_frequent,validation,0.405601,0.135200,0.333333,0.192374,0.234081


# **7. Baseline Interpretation**

This section records what the baseline results mean for the next modelling step.

In [8]:
# Summarise the strongest validation results.
best_category = category_results.iloc[0]
best_priority = priority_results.iloc[0]

findings = [
    f'Best queue/category baseline: {best_category["model"]} with macro F1 = {best_category["macro_f1"]:.4f}.',
    f'Best priority baseline: {best_priority["model"]} with macro F1 = {best_priority["macro_f1"]:.4f}.',
    'Both predictive tasks perform clearly above the dummy baseline, which supports the dataset replacement decision.',
    'Linear SVM is the strongest baseline for both queue routing and priority prediction in this run.',
]

for item in findings:
    print('-', item)

- Best queue/category baseline: linear_svm with macro F1 = 0.4932.
- Best priority baseline: linear_svm with macro F1 = 0.5489.
- Both predictive tasks perform clearly above the dummy baseline, which supports the dataset replacement decision.
- Linear SVM is the strongest baseline for both queue routing and priority prediction in this run.
